[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [8]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    # Replace this
    batch, seq, d = Q.shape
    device = Q.device

    # When you want a matrix where entry [i, j] depends on (i, j):

    # Create both index ranges as 1D arange tensors.
    # Unsqueeze one to (N, 1) — this becomes the "row axis."
    # Unsqueeze the other to (1, N) — this becomes the "column axis."
    # Combine them with whatever expression you want.

    # For our case, the expression is abs(i - j) <= w. For causal masking, 
    # it would be j <= i. For "everything within K steps in the past," 
    # it would be (i - j >= 0) & (i - j <= K). The pattern stays the same; only the predicate changes.

    i = torch.arange(seq, device=device).unsqueeze(1) # (seq, 1)
    j = torch.arange(seq, device=device).unsqueeze(0) # (1, seq)
    mask = (i - j).abs() <= window_size

    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)
    scores = scores.masked_fill(~mask, float('-inf'))

    attn = torch.softmax(scores, dim=-1)
    return attn @ V
    

In [6]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [7]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (0.8ms)
  ✅ [2/5] window_size=0 — only sees itself (0.4ms)
  ✅ [3/5] Large window equals full attention (3.4ms)
  ✅ [4/5] Distant tokens don't affect output (2.1ms)
  ✅ [5/5] Gradient flow (0.9ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (7.6ms total)
  Progress saved. Run status() to see your dashboard.

